# AE_Models

Notebook content includes EOS implementation and Thermal Resistance calculations.

In [ ]:
import sys
import os
from pathlib import Path

# Robustly add repo root to sys.path to access src/
current_dir = Path(os.getcwd()).resolve()
if (current_dir / "src").exists():
    repo_root = current_dir
elif (current_dir.parent / "src").exists():
    repo_root = current_dir.parent
else:
    # Fallback for when running in notebooks/ without direct evidence
    repo_root = Path(os.getcwd()).resolve().parent

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src import paths

paths.ensure_output_dirs()

OUT = paths.get_outputs_dir()
FIG = paths.get_figures_dir()
TAB = paths.get_tables_dir()
CACHE = paths.get_cache_dir()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from src import eos
from src import resistance

## 1. EOS Sanity Check

We calculate and plot the density of Hydrogen at 298 K for pressures ranging from 1 to 1000 bar.

In [ ]:
T_ref = 298.0  # K
P_range_bar = np.linspace(1, 1000, 100)
P_range_Pa = P_range_bar * 1e5

rho_vals = []
Z_vals = []

for P in P_range_Pa:
    rho = eos.rho_g_PR(P, T_ref)
    Z = eos.Z_PR(P, T_ref)
    rho_vals.append(rho)
    Z_vals.append(Z)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(P_range_bar, rho_vals)
plt.title(f'H2 Density at {T_ref} K')
plt.xlabel('Pressure (bar)')
plt.ylabel('Density (kg/m3)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(P_range_bar, Z_vals)
plt.title(f'Compressibility Factor Z at {T_ref} K')
plt.xlabel('Pressure (bar)')
plt.ylabel('Z')
plt.grid(True)

plt.tight_layout()
plt.show()

# Check specific values
P_check_bar = [1, 100, 700]
print(f"Values at {T_ref} K:")
for p_bar in P_check_bar:
    p_pa = p_bar * 1e5
    r = eos.rho_g_PR(p_pa, T_ref)
    z = eos.Z_PR(p_pa, T_ref)
    print(f"P = {p_bar} bar: rho = {r:.4f} kg/m3, Z = {z:.4f}")

## 2. Wall Resistance Comparison

We load the wall sets from configuration and compare thermal resistance.

In [ ]:
# Load config
config_path = os.path.join(paths.get_repo_root(), 'configs', 'wall_sets.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

wall_sets = config['wall_sets']

# Define dummy geometry
class Geometry:
    def __init__(self, A_int, A_ext):
        self.A_int = A_int
        self.A_ext = A_ext

geo = Geometry(A_int=1.0, A_ext=1.1)

# Calculate R_total for each set
results = {}
for name, wset in wall_sets.items():
    R = resistance.R_total(geo, wset)
    results[name] = R
    print(f"Wall Set: {name}, R_total: {R:.5f} K/W")

# Verification
if results['B_insulating'] > results['A_conductive']:
    print("\nSUCCESS: Insulating wall has higher resistance.")
else:
    print("\nFAILURE: Insulating wall should have higher resistance.")